In [ ]:
%load_ext autoreload
%autoreload 2

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (16, 8),
        'axes.labelsize': 'medium',
        'axes.titlesize': 'large',
        'xtick.labelsize': 'medium',
        'ytick.labelsize': 'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import datetime
import pytz

NYC = pytz.timezone('America/New_York')

import sys
sys.path.append('../../')

# SFRCalSpreadRV -- Specific Trade Analysis

Deep-dive into individual fly / spread trades.

**Focus:** U26 3M fly (M26/U26/Z26) -- our STIR trader's conviction trade.

In [ ]:
from BT.signals.sfr_cal_spread_rv import (
    SFRCalSpreadRVConfig,
    StructureType,
    STRUCTURE_LABELS,
    build_snapshot,
    load_rate_panel,
    compute_structure,
    compute_fly_curve,
    compute_zscore_ts,
    analyze_specific_fly,
)

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder

## 1. Load Data

In [ ]:
config = SFRCalSpreadRVConfig(
    n_contracts=12,
    zscore_window=40,
    vol_window=20,
)

curve_mdp = IRSwapsMDP(source=config.source)
ts_builder = TimeseriesBuilder()

start = NYC.localize(datetime.datetime(2025, 10, 1, 18, 0))
end = 'live'

rates = load_rate_panel(
    config, start=start, end=end,
    curve_mdp=curve_mdp, ts_builder=ts_builder,
)
print(f'Ladder: {list(rates.columns)}')
print(f'{rates.shape[0]} dates loaded')
rates.tail(3)

## 2. U26 3M Fly: M26/U26/Z26

Our STIR trader likes this fly. Let's see what the screener says.

In [ ]:
# Check which contracts are available
ladder = list(rates.columns)
print(f'Available contracts: {ladder}')

# Determine the U26 fly legs based on available contracts
# The U26 fly is M26/U26/Z26 (3M gap: front=M26, belly=U26, back=Z26)
u6_front = 'M26'
u6_belly = 'U26'
u6_back = 'Z26'

if all(c in ladder for c in [u6_front, u6_belly, u6_back]):
    u6 = analyze_specific_fly(rates, u6_front, u6_belly, u6_back, config)
    print(f'\n=== U26 3M Fly: {u6["trade"]} ===')
    print(f'  Level:        {u6["level_bp"]:+.2f} bp')
    print(f'  Prev Close:   {u6["prev_close_bp"]:+.2f} bp')
    print(f'  1d Change:    {u6["change_bp"]:+.2f} bp')
    print(f'  Z-Score:      {u6["zscore"]}')
    print(f'  Vol (ann):    {u6["vol_ann"]} bp')
    print(f'  Roll/Carry:   {u6["roll_bp"]} bp per quarter')
    print(f'  Risk-Adj Roll: {u6["risk_adj_roll"]}')
    print(f'  60d Mean:     {u6["mean_60d"]} bp')
    print(f'  60d Std:      {u6["std_60d"]} bp')
else:
    missing = [c for c in [u6_front, u6_belly, u6_back] if c not in ladder]
    print(f'WARNING: Contracts {missing} not in ladder. Adjust to available contracts.')
    # Try to use the first available 3M fly
    if len(ladder) >= 3:
        u6_front, u6_belly, u6_back = ladder[0], ladder[1], ladder[2]
        u6 = analyze_specific_fly(rates, u6_front, u6_belly, u6_back, config)
        print(f'\nUsing fallback: {u6["trade"]}')
        print(f'  Level:        {u6["level_bp"]:+.2f} bp')
        print(f'  Z-Score:      {u6["zscore"]}')
        print(f'  Roll/Carry:   {u6["roll_bp"]} bp per quarter')
        print(f'  Risk-Adj Roll: {u6["risk_adj_roll"]}')

In [ ]:
if 'u6' in dir() and 'timeseries' in u6:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [2, 1]})

    ts = u6['timeseries']
    zs = u6['zscore_ts']

    ax1.plot(ts.index, ts.values, color='tab:cyan', linewidth=1.5, label=f'{u6["trade"]} Level')
    if u6['mean_60d'] is not None:
        ax1.axhline(u6['mean_60d'], color='tab:orange', linestyle='--', linewidth=1, alpha=0.7,
                    label=f'60d Mean = {u6["mean_60d"]:.1f} bp')
    ax1.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax1.set_title(f'{u6["trade"]} Fly Level (bp)', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.fill_between(zs.index, 0, zs.values,
                     where=zs.values >= 0, color='tab:green', alpha=0.3)
    ax2.fill_between(zs.index, 0, zs.values,
                     where=zs.values < 0, color='tab:red', alpha=0.3)
    ax2.plot(zs.index, zs.values, color='tab:blue', linewidth=1)
    ax2.axhline(2, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    ax2.axhline(-2, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    ax2.axhline(0, color='gray', linestyle=':', linewidth=0.8)
    ax2.set_title(f'{u6["trade"]} Z-Score (40d rolling)', fontweight='bold')
    ax2.set_ylim(-3.5, 3.5)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 3. U26 Fly in Context: Full 3M Fly Curve

Where does U26 sit relative to the rest of the 3M fly curve?

In [ ]:
snap = build_snapshot(config, rates_panel=rates)

if StructureType.FLY_3M in snap.structures:
    fly3m = snap.structures[StructureType.FLY_3M]
    
    fig, ax = plt.subplots(figsize=(16, 7))
    x = range(len(fly3m.labels))
    
    # Plot curve
    ax.plot(x, fly3m.levels, 'o-', color='tab:cyan', linewidth=2, markersize=8, zorder=3)
    
    # Annotate each point
    for i, (lbl, lvl) in enumerate(zip(fly3m.labels, fly3m.levels)):
        if not np.isnan(lvl):
            ax.annotate(f'{lvl:.1f}', (i, lvl), textcoords='offset points',
                       xytext=(0, 12), ha='center', fontsize=9, color='red', fontweight='bold')
    
    # Highlight U26 fly if present
    u6_label = f'{u6_front}/{u6_belly}/{u6_back}'
    if u6_label in fly3m.labels:
        idx = fly3m.labels.index(u6_label)
        ax.scatter([idx], [fly3m.levels[idx]], color='yellow', s=200, zorder=4, edgecolor='red', linewidth=2,
                  label=f'U26 Fly = {fly3m.levels[idx]:.1f} bp')
    
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(fly3m.labels, rotation=45, ha='right', fontsize=9)
    ax.set_title('3M Microfly Curve (bp) -- U26 Highlighted', fontweight='bold', fontsize=14)
    ax.set_ylabel('Fly Level (bp)')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Show the full table
    print('\n=== 3M Fly Full Table ===')
    display(fly3m.to_dataframe())

## 4. Nearby Fly Comparison

Compare U26 fly to its neighbors to assess relative attractiveness.

In [ ]:
# Analyze a few flies around U26 for comparison
comparison_flies = []
for i in range(len(ladder) - 2):
    front, belly, back = ladder[i], ladder[i+1], ladder[i+2]
    try:
        result = analyze_specific_fly(rates, front, belly, back, config)
        comparison_flies.append({
            'Fly': result['trade'],
            'Level (bp)': result['level_bp'],
            'Chg (bp)': result['change_bp'],
            'Z-Score': result['zscore'],
            'Vol (ann)': result['vol_ann'],
            'Roll (bp)': result['roll_bp'],
            'Risk-Adj Roll': result['risk_adj_roll'],
        })
    except Exception as e:
        pass

if comparison_flies:
    comp_df = pd.DataFrame(comparison_flies).set_index('Fly')
    
    # Style: highlight the U26 row
    print('=== All 3M Microflies Detailed Comparison ===')
    display(comp_df.round(2))

## 5. 6M and 12M Flies Through U26

Wider gap flies centered on U26 to see the bigger picture.

In [ ]:
# 6M fly centered on U26: if U26 is belly, wings are 6M apart
# i.e., Z25/U26/M27 (if available)
wider_results = []

if u6_belly in ladder:
    belly_idx = ladder.index(u6_belly)
    
    # 3M fly (gap=1)
    if belly_idx >= 1 and belly_idx + 1 < len(ladder):
        try:
            r = analyze_specific_fly(rates, ladder[belly_idx-1], ladder[belly_idx], ladder[belly_idx+1], config)
            wider_results.append({'Type': '3M Fly', **{k: r[k] for k in ['trade', 'level_bp', 'zscore', 'roll_bp', 'risk_adj_roll']}})
        except: pass
    
    # 6M fly (gap=2)
    if belly_idx >= 2 and belly_idx + 2 < len(ladder):
        try:
            r = analyze_specific_fly(rates, ladder[belly_idx-2], ladder[belly_idx], ladder[belly_idx+2], config)
            wider_results.append({'Type': '6M Fly', **{k: r[k] for k in ['trade', 'level_bp', 'zscore', 'roll_bp', 'risk_adj_roll']}})
        except: pass

    # 9M fly (gap=3)
    if belly_idx >= 3 and belly_idx + 3 < len(ladder):
        try:
            r = analyze_specific_fly(rates, ladder[belly_idx-3], ladder[belly_idx], ladder[belly_idx+3], config)
            wider_results.append({'Type': '9M Fly', **{k: r[k] for k in ['trade', 'level_bp', 'zscore', 'roll_bp', 'risk_adj_roll']}})
        except: pass

    # 12M fly (gap=4)
    if belly_idx >= 4 and belly_idx + 4 < len(ladder):
        try:
            r = analyze_specific_fly(rates, ladder[belly_idx-4], ladder[belly_idx], ladder[belly_idx+4], config)
            wider_results.append({'Type': '12M Fly', **{k: r[k] for k in ['trade', 'level_bp', 'zscore', 'roll_bp', 'risk_adj_roll']}})
        except: pass

if wider_results:
    wider_df = pd.DataFrame(wider_results).set_index('Type')
    print(f'=== Flies Centered on {u6_belly} at Various Gaps ===')
    display(wider_df)
else:
    print(f'{u6_belly} not available for wider fly analysis')

## 6. Carry/Roll Table (Barnes Style)

For each microfly: current level, where it rolls to, and the implied carry.

In [ ]:
if StructureType.FLY_3M in snap.structures:
    fly3m = snap.structures[StructureType.FLY_3M]
    carry_rows = []
    for i, label in enumerate(fly3m.labels):
        row = {
            'Microfly': label,
            'Level (bp)': round(fly3m.levels[i], 2),
            'Rolls To': fly3m.labels[i-1] if i > 0 else '--',
            'Target (bp)': round(fly3m.levels[i-1], 2) if i > 0 and not np.isnan(fly3m.levels[i-1]) else None,
            'Carry (bp)': round(fly3m.rolls[i], 2) if not np.isnan(fly3m.rolls[i]) else None,
            'Direction': ('BUY belly' if fly3m.rolls[i] > 0 else 'SELL belly') if not np.isnan(fly3m.rolls[i]) else '--',
        }
        carry_rows.append(row)
    
    carry_df = pd.DataFrame(carry_rows).set_index('Microfly')
    print('=== 3M Microfly Carry Table (Barnes Style) ===')
    print('Positive carry = earn if long belly (buy the fly)')
    print('Negative carry = earn if short belly (sell the fly)\n')
    display(carry_df)

## 7. Summary & Conviction Assessment

In [ ]:
print('=' * 70)
print('CONVICTION ASSESSMENT')
print('=' * 70)

if 'u6' in dir():
    print(f'\nTrade: {u6["trade"]} (3M Microfly)')
    print(f'Current Level: {u6["level_bp"]:+.2f} bp')
    print(f'1d Change:     {u6["change_bp"]:+.2f} bp')
    print()
    
    # Conviction signals
    signals = []
    
    zs = u6['zscore']
    if zs is not None:
        if abs(zs) > 2:
            signals.append(f'STRONG: Z-score = {zs:+.2f} (extreme -- potential mean reversion)')
        elif abs(zs) > 1:
            signals.append(f'MODERATE: Z-score = {zs:+.2f} (elevated but not extreme)')
        else:
            signals.append(f'NEUTRAL: Z-score = {zs:+.2f} (within normal range)')
    
    roll = u6['roll_bp']
    if roll is not None:
        if roll > 0:
            signals.append(f'POSITIVE CARRY: Roll = {roll:+.2f} bp/qtr (long belly earns carry)')
        elif roll < 0:
            signals.append(f'NEGATIVE CARRY: Roll = {roll:+.2f} bp/qtr (long belly pays carry)')
    
    radj = u6['risk_adj_roll']
    if radj is not None:
        if abs(radj) > 1:
            signals.append(f'ATTRACTIVE risk-adj roll = {radj:+.2f} (>1 is strong)')
        else:
            signals.append(f'MODERATE risk-adj roll = {radj:+.2f}')
    
    vol = u6['vol_ann']
    if vol is not None:
        signals.append(f'Realized vol = {vol:.1f} bp (annualized)')
    
    for s in signals:
        print(f'  * {s}')
    
    print()
    if zs is not None and roll is not None:
        if zs < -1.5 and roll > 0:
            print('VERDICT: BUY the fly -- cheap on z-score + positive carry')
        elif zs > 1.5 and roll < 0:
            print('VERDICT: SELL the fly -- rich on z-score + negative carry')
        elif abs(zs) < 0.5:
            print('VERDICT: HOLD / MONITOR -- fair value, wait for better entry')
        else:
            direction = 'BUY' if zs < 0 else 'SELL'
            carry_align = 'ALIGNED' if (zs < 0 and roll > 0) or (zs > 0 and roll < 0) else 'OPPOSED'
            print(f'VERDICT: Lean {direction} but carry is {carry_align} -- scale in cautiously')
else:
    print('U6 fly data not available -- see above for available contracts')